# 02. Baseline: ConvNeXt-Tiny на 80k сабсеті

Мета цього ноутбука — довести пайплайн навчання end-to-end і зафіксувати перші чесні метрики
моделі, яка за фото автомобіля одночасно передбачає рік випуску (`year`) та ціну (`price_gbp`,
у лог-просторі). Backbone — **ConvNeXt-Tiny**, навчання у дві стадії:

- **Stage 1** — backbone заморожений, навчається лише голова (multi-task head для року і ціни).
- **Stage 2** — backbone розморожено, fine-tune усієї мережі з меншим learning rate.

Дані — 80k-сабсет із маніфесту (`scripts/build_manifest.py`), поділений на train/val/test/holdout,
де holdout навмисно містить моделі авто, не бачені під час навчання (перевірка generalization).
Тут ми не перенавчаємо нічого — лише читаємо вже збережені результати з `results/convnext/`.


In [ ]:
import json

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

metrics = pd.read_csv("../results/convnext/metrics.csv")

# Наскрізна нумерація епох для суцільної осі X: Stage 1 (epochs 0..4) займає кроки 0..4,
# Stage 2 (epochs 0..14) продовжує нумерацію далі (кроки 5..19).
n_stage1_epochs = int(metrics.loc[metrics["stage"] == 1, "epoch"].max() + 1)
n_stage2_epochs = int(metrics.loc[metrics["stage"] == 2, "epoch"].max() + 1)
metrics["global_step"] = metrics["epoch"] + (metrics["stage"] == 2) * n_stage1_epochs
stage_boundary = n_stage1_epochs - 0.5

print(f"Stage 1 (заморожений backbone): {n_stage1_epochs} епох")
print(f"Stage 2 (fine-tune): {n_stage2_epochs} епох")
metrics.tail(8)


## Функція втрат (loss): train vs val

Дивимось на loss на обох стадіях одразу, з вертикальною лінією на межі stage 1 → stage 2.
Очікуваний патерн: помітний стрибок вниз одразу після розморожування backbone, оскільки
модель отримує змогу підлаштовувати самі візуальні ознаки під задачу, а не тільки голову.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for split, color in [("train", "steelblue"), ("val", "darkorange")]:
    sub = metrics[metrics["split"] == split].sort_values("global_step")
    ax.plot(sub["global_step"], sub["loss"], marker="o", ms=3, color=color, label=split)
ax.axvline(stage_boundary, ls="--", color="gray", label="Stage 1 → Stage 2")
ax.set_title("ConvNeXt-Tiny: функція втрат (loss) по епохах")
ax.set_xlabel("Епоха (наскрізна нумерація: stage 1 + stage 2)")
ax.set_ylabel("Loss")
ax.legend()
plt.tight_layout()
plt.show()


## Помилка року (val `year_mae`)

`year_mae` тут вже в роках (не в стандартизованих одиницях), тож число одразу інтерпретується:
"в середньому модель помиляється на X років". Якщо fine-tune (stage 2) справді допомагає,
крива має помітно просісти саме після межі стадій, а не плавно продовжувати тренд stage 1.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sub = metrics[metrics["split"] == "val"].sort_values("global_step")
ax.plot(sub["global_step"], sub["year_mae"], marker="o", ms=3, color="darkorange")
ax.axvline(stage_boundary, ls="--", color="gray", label="Stage 1 → Stage 2")
ax.set_title("ConvNeXt-Tiny: помилка року (val year_mae) по епохах")
ax.set_xlabel("Епоха (наскрізна нумерація)")
ax.set_ylabel("MAE року, років")
ax.legend()
plt.tight_layout()
plt.show()


## Помилка ціни в лог-просторі (val `price_log_mae`)

`price_log_mae` — це MAE у просторі логарифма ціни (вже де-стандартизоване). Значення тут
менш інтуїтивне, ніж роки, але тренд читається так само: чи стадія fine-tune дає додатковий
виграш над замороженим backbone, чи модель на цій стадії вже виходить на плато.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sub = metrics[metrics["split"] == "val"].sort_values("global_step")
ax.plot(sub["global_step"], sub["price_log_mae"], marker="o", ms=3, color="seagreen")
ax.axvline(stage_boundary, ls="--", color="gray", label="Stage 1 → Stage 2")
ax.set_title("ConvNeXt-Tiny: помилка ціни в лог-просторі (val price_log_mae) по епохах")
ax.set_xlabel("Епоха (наскрізна нумерація)")
ax.set_ylabel("MAE, лог-одиниці ціни")
ax.legend()
plt.tight_layout()
plt.show()


## Підсумкові метрики: val / test / holdout

Тепер дивимось не на криві навчання, а на фінальну оцінку останньої (fine-tuned) моделі на
трьох сплітах. `holdout` — головний тест на generalization: він містить моделі авто, яких
модель під час навчання взагалі не бачила.


In [ ]:
with open("../results/convnext/eval_metrics.json") as f:
    eval_metrics = json.load(f)

summary_cols = ["mae_years", "mae_log", "mape", "r2_year", "r2_price_log", "within_brand_corr_mean"]
summary_table = pd.DataFrame(
    {split: {c: eval_metrics[split][c] for c in summary_cols} for split in ["val", "test", "holdout"]}
).T
summary_table


In [ ]:
val_mae = eval_metrics["val"]["mae_years"]
holdout_mae = eval_metrics["holdout"]["mae_years"]
gap_years = holdout_mae - val_mae
gap_pct = 100 * gap_years / val_mae

display(Markdown(
    f"**Висновок.** Пайплайн відпрацював наскрізно: за {n_stage1_epochs} епох із замороженим "
    f"backbone і {n_stage2_epochs} епох fine-tune модель виходить на val MAE року ≈ "
    f"**{val_mae:.2f} років** (MAPE ціни ≈ {eval_metrics['val']['mape']:.1f}%). Це перші чесні "
    f"числа baseline — ще без підбору гіперпараметрів чи аугментацій.\n\n"
    f"На `holdout` (моделі авто, не бачені під час навчання) MAE року зростає до "
    f"**{holdout_mae:.2f} років** — розрив ≈ {gap_years:.2f} років ({gap_pct:.0f}% відносно val). "
    f"Це очікуваний generalization gap: модель добре інтерполює між бренд/моделями, які бачила, "
    f"але гірше екстраполює на повністю нові моделі авто. Саме тому holdout — головний орієнтир "
    f"для чесної оцінки, а не val."
))
